In [10]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
#MODEL = "gpt-3.5-turbo"
#MODEL = "mixtral:8x7b"
MODEL = "llama2"
from langchain_community.llms import Ollama
#from langchain_openai.chat_models import ChatOpenAI
from langchain_community.embeddings import OllamaEmbeddings
#from langchain_openai.embeddings import OpenAIEmbeddings

if MODEL.startswith("gpt"):
    model = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model=MODEL)
    embeddings = OpenAIEmbeddings()
else:
    model = Ollama(model=MODEL)
    embeddings = OllamaEmbeddings(model=MODEL)

model.invoke("Tell me a joke")

C:\Users\jatro\AppData\Local\Temp\ipykernel_2860\673762214.py:19: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  model = Ollama(model=MODEL)
C:\Users\jatro\AppData\Local\Temp\ipykernel_2860\673762214.py:20: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model=MODEL)


"\nWhy don't scientists trust atoms? Because they make up everything! 😂"

In [11]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = model | parser 
chain.invoke("Tell me a joke")

"\nWhy don't scientists trust atoms? Because they make up everything! 😂"

In [12]:
from langchain.prompts import PromptTemplate

template = """
Answer the question based on the context below. If you can't 
answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt.format(context="Here is some context", question="Here is a question")

'\nAnswer the question based on the context below. If you can\'t \nanswer the question, reply "I don\'t know".\n\nContext: Here is some context\n\nQuestion: Here is a question\n'

In [13]:
chain = prompt | model | parser

chain.invoke({"context": "My parents named me Santiago", "question": "What's your name'?"})

' Great! Based on the context you provided, my answer is: Santiago'

In [15]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r"C:\Users\jatro\Dev\RAGwithPDFs\venv\data\Mech eng textbook .pdf")
pages = loader.load_and_split()
pages

Ignoring wrong pointing object 5448 0 (offset 0)
Ignoring wrong pointing object 9406 0 (offset 0)
Ignoring wrong pointing object 9609 0 (offset 0)
Ignoring wrong pointing object 9724 0 (offset 0)


[Document(metadata={'producer': 'iOS Version 18.1.1 (Build 22B91) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250219110228Z00'00'", 'moddate': "D:20250219110228Z00'00'", 'source': 'C:\\Users\\jatro\\Dev\\RAGwithPDFs\\venv\\data\\Mech eng textbook .pdf', 'total_pages': 383, 'page': 0, 'page_label': '1'}, page_content='Copyright David JC MacKay 2009. This electronic copy is provided, free, for personal use only. See www.withouthotair.com.\nSustainable Energy — without the hot air\nVersion 3.5.2. November 3, 2008.\nThis Cover-sheet must not appear in the printed book.high-resolution edition.'),
 Document(metadata={'producer': 'iOS Version 18.1.1 (Build 22B91) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250219110228Z00'00'", 'moddate': "D:20250219110228Z00'00'", 'source': 'C:\\Users\\jatro\\Dev\\RAGwithPDFs\\venv\\data\\Mech eng textbook .pdf', 'total_pages': 383, 'page': 1, 'page_label': '2'}, page_content='Copyright David JC MacKay 2009. This electronic 

In [17]:
from langchain_community.vectorstores import DocArrayInMemorySearch

vectorstore = DocArrayInMemorySearch.from_documents(pages, embedding=embeddings)

c:\Users\jatro\Dev\RAGwithPDFs\venv\Lib\site-packages\pydantic\_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [18]:
retriever = vectorstore.as_retriever()
retriever.invoke("sub-saharan Africa")

[Document(metadata={'producer': 'iOS Version 18.1.1 (Build 22B91) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250219110228Z00'00'", 'moddate': "D:20250219110228Z00'00'", 'source': 'C:\\Users\\jatro\\Dev\\RAGwithPDFs\\venv\\data\\Mech eng textbook .pdf', 'total_pages': 383, 'page': 356, 'page_label': '357'}, page_content='Copyright David JC MacKay 2009. This electronic copy is provided, free, for personal use only. See www.withouthotair.com.\nList of web links\nThis section lists the full links corresponding to each of the tiny URLs\nmentioned in the text. Each item starts with the page number on which\nthe tiny URL was mentioned. See also http://tinyurl.com/yh8xse (or www.\ninference.phy.cam.ac.uk/sustainable/book/tex/cft.url.html )f o rac l i c k a b l e\npage with all URLs in this book.\nIf you ﬁnd a URL doesn’t work any more, you may be able to ﬁnd the\npage on the Wayback Machine internet archive [f754].\npt i n y U R L F u l l w e b l i n k .\n18 ydoobr www.bbc.co

In [19]:
from operator import itemgetter

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
    }
    | prompt
    | model
    | parser
)

In [20]:
questions = [
    "What is the purpose of the course?",
    "How many hours of live sessions?",
    "How many coding assignments are there in the program?",
    "Is there a program certificate upon completion?",
    "What programming language will be used in the program?",
    "How much does the program cost?",
]

for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {chain.invoke({'question': question})}")
    print()

Question: What is the purpose of the course?
Answer: The purpose of the course is to educate students on the domestic consumption stack, which includes heating, cooling, cooking, and powering appliances in homes and workplaces. The course covers topics such as energy consumption, renewable energy sources, and low-grade thermal energy. The author estimates the average daily energy consumption per person for space heating, water, and cooking, and provides national statistics for comparison. The course also includes notes and further reading on page numbers related to the topic of electric ovens and their maximum power consumption.

Question: How many hours of live sessions?
Answer: Based on the document provided, there are 383 pages in the document, and the page number 380 corresponds to the page where the information about live sessions is mentioned. Therefore, there are approximately 380 live sessions mentioned in the document.

Question: How many coding assignments are there in the pr

In [21]:
chain.batch([{"question": q} for q in questions])

['The purpose of the course is to educate students on the topic of energy consumption and production, with a focus on the domestic sector. The course covers various aspects of energy use in homes, including heating, cooling, cooking, and powering appliances. It also provides estimates of energy consumption per person for different sectors, such as domestic, service, and workplace.\n\nThe course aims to:\n\n1. Provide students with an understanding of the different types of energy consumed in homes.\n2. Help students identify areas where energy can be saved or improved.\n3. Educate students on the importance of energy efficiency and renewable energy sources.\n4. Encourage students to think critically about their own energy consumption habits and how they can contribute to a more sustainable future.\n5. Develop skills in critical thinking, problem-solving, and decision-making related to energy use.\n6. Provide students with the knowledge and tools needed to make informed choices about en

In [22]:
for s in chain.stream({"question": "What is the purpose of the course?"}):
    print(s, end="", flush=True)

The purpose of this course appears to be to educate students on various aspects of energy consumption and production, including renewable energy sources, energy efficiency, and the role of individuals in reducing energy consumption. The course also aims to provide students with practical skills and knowledge to help them understand how to reduce their own energy consumption and contribute to a more sustainable future.

Throughout the course, the instructor provides various examples and estimates to support their arguments, such as their own domestic gas consumption data, national statistics on energy consumption, and product information on ovens and other appliances. The course materials also include notes and further reading, which provide additional context and resources for students to explore.

Overall, the purpose of this course seems to be to inspire and empower students to take action towards reducing their own energy consumption and contributing to a more sustainable future.